# P&L 04 — hedged versus unhedged

Run four P&L combinations for each exposure scenario from the November 4, 2024 U.S. close:

- unhedged, one-day election-reaction endpoint (November 6)
- hedged, one-day election-reaction endpoint
- unhedged, five-trading-day lag endpoint (November 12)
- hedged, five-trading-day lag endpoint

The hedge is the TAN position plus the positive Trump-Yes share quantity constructed in `hedge03`. Because historical quotes are unavailable, the November 4 ask is proxied by the CLOB midpoint at 20:00 UTC plus the empirical effective half-spread estimated near that close.

In [ ]:
from pathlib import Path

import pandas as pd

HEDGE_SCENARIOS_PATH = Path("data/hedge03_scenarios.csv")
EXECUTION_COSTS_PATH = Path("data/hedge03_execution_costs.csv")
ETF_DATA_PATH = Path("data/tan_spy_daily_2016_2024.csv")
POLYMARKET_PATH = Path("data/polymarket_trump_2024_yes_1min.csv")
PATH_OUTPUT = Path("data/pnl04_daily_paths.csv")
SUMMARY_OUTPUT = Path("data/pnl04_four_path_summary.csv")

ENTRY_DATE = pd.Timestamp("2024-11-04")
ENTRY_TIMESTAMP = pd.Timestamp("2024-11-04T20:00:00Z")
ONE_DAY_ENDPOINT = pd.Timestamp("2024-11-06")
FIVE_DAY_ENDPOINT = pd.Timestamp("2024-11-12")

In [ ]:
hedges = pd.read_csv(HEDGE_SCENARIOS_PATH)
execution_costs = pd.read_csv(EXECUTION_COSTS_PATH)
etf_prices = pd.read_csv(ETF_DATA_PATH, parse_dates=["Date"])
polymarket = pd.read_csv(POLYMARKET_PATH, parse_dates=["timestamp_utc"]).sort_values("timestamp_utc")

# TAN adjusted closes determine the equity P&L path.
tan_prices = (
    etf_prices.loc[etf_prices["Ticker"].eq("TAN"), ["Date", "Adj Close"]]
    .set_index("Date")
    .sort_index()
)
path_dates = tan_prices.loc[ENTRY_DATE:FIVE_DAY_ENDPOINT].index
entry_tan_price = float(tan_prices.loc[ENTRY_DATE, "Adj Close"])

# Ask proxy at the U.S. close: latest midpoint plus empirical effective half-spread.
midpoint_at_entry = float(
    polymarket.set_index("timestamp_utc")["price"].asof(ENTRY_TIMESTAMP)
)
effective_half_spreads = execution_costs["b_effective_half_spread"].dropna().unique()
if len(effective_half_spreads) != 1:
    raise RuntimeError("Expected one effective half-spread estimate.")
effective_half_spread = float(effective_half_spreads[0])
ask_at_entry = midpoint_at_entry + effective_half_spread

assert path_dates[0] == ENTRY_DATE
assert path_dates[-1] == FIVE_DAY_ENDPOINT
assert 0 < ask_at_entry < 1

{
    "entry_TAN_adjusted_close": entry_tan_price,
    "Polymarket_midpoint_at_20UTC": midpoint_at_entry,
    "effective_half_spread": effective_half_spread,
    "ask_proxy": ask_at_entry,
}

In [ ]:
polymarket_daily = (
    polymarket.assign(
        Date=polymarket["timestamp_utc"].dt.floor("D").dt.tz_localize(None)
    )
    .groupby("Date")["price"]
    .last()
)

contract_marks = pd.Series(index=path_dates, dtype=float, name="contract_mark")
contract_marks.loc[ENTRY_DATE] = ask_at_entry  # P&L starts at zero at execution.
for date in path_dates[path_dates > ENTRY_DATE]:
    if date < ONE_DAY_ENDPOINT:
        contract_marks.loc[date] = polymarket_daily.loc[date]
    else:
        contract_marks.loc[date] = 1.0  # Trump Yes resolved.

path_frames = []
for row in hedges.itertuples(index=False):
    tan_units = row.Q / entry_tan_price
    scenario_path = pd.DataFrame(
        {
            "scenario": row.scenario,
            "Date": path_dates,
            "Q": row.Q,
            "TAN_units": tan_units,
            "TAN_adjusted_close": tan_prices.loc[path_dates, "Adj Close"].to_numpy(),
            "yes_contracts": row.yes_shares_to_buy,
            "contract_mark": contract_marks.to_numpy(),
        }
    )
    scenario_path["unhedged_PnL"] = scenario_path["TAN_units"] * (
        scenario_path["TAN_adjusted_close"] - entry_tan_price
    )
    scenario_path["contract_PnL"] = scenario_path["yes_contracts"] * (
        scenario_path["contract_mark"] - ask_at_entry
    )
    scenario_path["hedged_PnL"] = (
        scenario_path["unhedged_PnL"] + scenario_path["contract_PnL"]
    )
    path_frames.append(scenario_path)

pnl_paths = pd.concat(path_frames, ignore_index=True)
pnl_paths.to_csv(PATH_OUTPUT, index=False)
pnl_paths

In [ ]:
endpoint_labels = {
    ONE_DAY_ENDPOINT: "1-day reaction (Nov 6)",
    FIVE_DAY_ENDPOINT: "5-day lag (Nov 6–12)",
}
summary_rows = []

for scenario in hedges["scenario"]:
    scenario_path = pnl_paths.loc[pnl_paths["scenario"].eq(scenario)].set_index("Date")
    for endpoint, horizon in endpoint_labels.items():
        row = scenario_path.loc[endpoint]
        for hedge_status, pnl_column in [
            ("Unhedged", "unhedged_PnL"),
            ("Hedged", "hedged_PnL"),
        ]:
            summary_rows.append(
                {
                    "scenario": scenario,
                    "horizon": horizon,
                    "endpoint": endpoint.date().isoformat(),
                    "hedge_status": hedge_status,
                    "PnL": row[pnl_column],
                    "PnL_pct_of_Q": row[pnl_column] / row["Q"],
                    "TAN_PnL": row["unhedged_PnL"],
                    "contract_PnL": 0.0 if hedge_status == "Unhedged" else row["contract_PnL"],
                }
            )

four_path_summary = pd.DataFrame(summary_rows)
four_path_summary.to_csv(SUMMARY_OUTPUT, index=False)

print(f"Entry ask proxy: {ask_at_entry:.6f}")
print(f"Saved daily paths to {PATH_OUTPUT.resolve()}")
print(f"Saved endpoint summary to {SUMMARY_OUTPUT.resolve()}")
four_path_summary

## Endpoint results

The linear hedge produces the same percentage returns at both scales:

- **One-day election-reaction endpoint (Nov 6)**
  - Unhedged: **−9.32% of Q**
  - Hedged: **+0.53% of Q**
- **Five-day lag endpoint (Nov 12)**
  - Unhedged: **−15.48% of Q**
  - Hedged: **−5.63% of Q**

Retail dollar P&L:
- One-day: **−$9,323 unhedged** versus **+$531 hedged**
- Five-day: **−$15,482 unhedged** versus **−$5,628 hedged**

Institutional dollar P&L:
- One-day: **−$4.662M unhedged** versus **+$0.265M hedged**
- Five-day: **−$7.741M unhedged** versus **−$2.814M hedged**

The hedge offsets the immediate election move but not the full delayed TAN decline. Institutional results are mechanical mark-to-resolution P&L and exclude the substantial feasibility and impact constraints documented in `hedge03`.

## Hedged-residual decomposition

The decomposition is additive in return space and uses market-adjusted TAN performance so the broad SPY move is not mislabeled as hedge error:

\[
\text{hedged residual}_{5d}
= \underbrace{(R^{MA}_{1d}-\widehat R)}_{\Delta E\text{ estimation error}}
+ \underbrace{(R^{MA}_{5d}-R^{MA}_{1d})}_{p_s\ne p\text{ convergence proxy}}
- \underbrace{\text{Phase 3 friction}}_{\text{spread + impact}}.
\]

The earlier normalized Phase 2 error, `(realized − predicted) / predicted`, is useful for forecast evaluation but cannot be added directly to P&L. Here its dollar-equivalent contribution is `realized − predicted`.

In [ ]:
PHASE2_PREDICTION_PATH = Path("data/outofsamplevalidation02_nov6_prediction.csv")
PHASE3_FRICTION_PATH = Path("data/hedge03_final_friction_table.csv")
DECOMPOSITION_PATH = Path("data/pnl04_hedged_residual_decomposition.csv")

phase2_prediction = pd.read_csv(PHASE2_PREDICTION_PATH)
phase3_friction = pd.read_csv(PHASE3_FRICTION_PATH).set_index("scenario")
predicted_move = float(phase2_prediction.loc[0, "predicted_TAN_market_adjusted_move"])

adjusted_close = (
    etf_prices.pivot(index="Date", columns="Ticker", values="Adj Close")
    .sort_index()
)
market_adjusted_moves = {}
for endpoint in [ONE_DAY_ENDPOINT, FIVE_DAY_ENDPOINT]:
    returns = adjusted_close.loc[endpoint, ["TAN", "SPY"]] / adjusted_close.loc[
        ENTRY_DATE, ["TAN", "SPY"]
    ] - 1
    market_adjusted_moves[endpoint] = returns["TAN"] - returns["SPY"]

realized_ma_1day = market_adjusted_moves[ONE_DAY_ENDPOINT]
realized_ma_5day = market_adjusted_moves[FIVE_DAY_ENDPOINT]
delta_E_error_component = realized_ma_1day - predicted_move
convergence_component = realized_ma_5day - realized_ma_1day

{
    "predicted_market_adjusted_move": predicted_move,
    "realized_market_adjusted_1day": realized_ma_1day,
    "realized_market_adjusted_5day": realized_ma_5day,
    "delta_E_error_component": delta_E_error_component,
    "p_s_not_equal_p_convergence_component": convergence_component,
}

In [ ]:
decomposition_rows = []
for scenario in hedges["scenario"]:
    friction_component = -phase3_friction.loc[scenario, "total_friction_bps_of_Q"] / 10_000
    decomposed_residual = (
        delta_E_error_component + convergence_component + friction_component
    )
    direct_residual = realized_ma_5day - predicted_move + friction_component
    assert abs(decomposed_residual - direct_residual) < 1e-12

    decomposition_rows.extend(
        [
            {
                "scenario": scenario,
                "component": "Phase 2: ΔE estimation error",
                "contribution_pct_Q": 100 * delta_E_error_component,
            },
            {
                "scenario": scenario,
                "component": "p_s ≠ p convergence (5-day minus 1-day)",
                "contribution_pct_Q": 100 * convergence_component,
            },
            {
                "scenario": scenario,
                "component": "Phase 3: friction",
                "contribution_pct_Q": 100 * friction_component,
            },
            {
                "scenario": scenario,
                "component": "Total hedged residual",
                "contribution_pct_Q": 100 * decomposed_residual,
            },
        ]
    )

decomposition = pd.DataFrame(decomposition_rows)
decomposition.to_csv(DECOMPOSITION_PATH, index=False)

print(f"Saved decomposition to {DECOMPOSITION_PATH.resolve()}")
decomposition

## Decomposition result

Contributions to the five-day market-adjusted hedged residual, in percentage points of Q:

- **ΔE estimation error:** **−3.18 pp** for both scenarios
- **p_s ≠ p convergence / lag:** **−7.19 pp** for both scenarios
- **Friction:** **−0.018 pp retail**; **−0.210 pp institutional**
- **Total residual:** **−10.38% retail**; **−10.57% institutional**

The convergence term is the largest component. Estimation error is second. Friction is negligible for retail but visibly worsens the institutional residual, consistent with the Phase 3 scale verdict.

These are market-adjusted residuals. The raw TAN-plus-contract P&L paths above also contain the broad SPY return. All phases now use the same 20:00 UTC entry timestamp: the forecast and decomposition use its midpoint probability, while the raw hedge P&L uses the corresponding midpoint-plus-half-spread ask proxy.

## Centerpiece deliverable

The figure reports P&L as a share of Q at retail scale; before institutional execution impact, percentage paths are mechanically scale-invariant. Color identifies hedge status and line style identifies the evaluation horizon. The one-day lines stop on November 6, while the five-day lines continue through November 12.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

FIGURE_PATH = Path("figures/pnl04_four_pnl_paths.png")
FIGURE_PATH.parent.mkdir(exist_ok=True)

plot_data = pnl_paths.loc[pnl_paths["scenario"].eq("Q_retail")].set_index("Date").copy()
plot_data["unhedged_pct_Q"] = plot_data["unhedged_PnL"] / plot_data["Q"]
plot_data["hedged_pct_Q"] = plot_data["hedged_PnL"] / plot_data["Q"]
one_day_plot = plot_data.loc[:ONE_DAY_ENDPOINT]

fig, ax = plt.subplots(figsize=(10, 6))

# Five-day paths first; solid one-day paths overlay their shared prefix.
ax.plot(
    plot_data.index,
    plot_data["unhedged_pct_Q"],
    color="#C23B22",
    linestyle="--",
    marker="o",
    linewidth=2,
    label="Unhedged · 5-day",
)
ax.plot(
    plot_data.index,
    plot_data["hedged_pct_Q"],
    color="#1565C0",
    linestyle="--",
    marker="o",
    linewidth=2,
    label="Hedged · 5-day",
)
ax.plot(
    one_day_plot.index,
    one_day_plot["unhedged_pct_Q"],
    color="#C23B22",
    linestyle="-",
    marker="o",
    linewidth=3,
    label="Unhedged · 1-day",
)
ax.plot(
    one_day_plot.index,
    one_day_plot["hedged_pct_Q"],
    color="#1565C0",
    linestyle="-",
    marker="o",
    linewidth=3,
    label="Hedged · 1-day",
)

ax.set_title("Election hedge P&L from November 4 close")
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative P&L (% of Q)")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xticks(plot_data.index)
ax.tick_params(axis="x", rotation=35)
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
fig.savefig(FIGURE_PATH, dpi=300, bbox_inches="tight")

print(f"Saved centerpiece figure to {FIGURE_PATH.resolve()}")
plt.show()

In [ ]:
DECOMPOSITION_TABLE_PATH = Path("data/pnl04_decomposition_table.csv")
component_order = [
    "Phase 2: ΔE estimation error",
    "p_s ≠ p convergence (5-day minus 1-day)",
    "Phase 3: friction",
    "Total hedged residual",
]

decomposition_table = (
    decomposition.pivot(
        index="component",
        columns="scenario",
        values="contribution_pct_Q",
    )
    .reindex(component_order)
    .rename(columns={"Q_retail": "Retail (% of Q)", "Q_inst": "Institutional (% of Q)"})
)
decomposition_table.to_csv(DECOMPOSITION_TABLE_PATH, index_label="Component")

print(f"Saved decomposition table to {DECOMPOSITION_TABLE_PATH.resolve()}")
decomposition_table

## Deliverable files

- Four-line centerpiece figure: `figures/pnl04_four_pnl_paths.png`
- Decomposition table: `data/pnl04_decomposition_table.csv`

Together they show that the hedge neutralizes the immediate election move, but delayed convergence dominates the five-day residual; institutional friction adds a smaller but visible scale penalty.